In [ ]:
import pandas as pd
from pathlib import Path
import seaborn as sns
import scipy
from matplotlib import pyplot as plt

In [ ]:
breakupdata = Path("../data/breakupdata/derived/breakupDate_cleaned.csv")
icedata = Path("../data/predictors/ice_thickness_POR_BobBusey.csv")
outdir = Path("../data/working/")

### Nenana only

In [ ]:
breakup = pd.read_csv(breakupdata, skiprows=3, index_col=0)
breakup = breakup[breakup.siteID=='Tanana River at Nenana']
ice = pd.read_csv(icedata).set_index('year')
ice.columns = ['thick']
ice

In [ ]:
breakup = breakup[['year', 'JulianDay']].set_index('year')
breakup

In [ ]:
sns.scatterplot(data=ice, x='year', y='thick', )

In [ ]:
fig = plt.figure(figsize=(9, 6))
sns.regplot(data=breakup.join(ice).dropna(), x='thick', y='JulianDay')
plt.title("Tanana River at Nenana: Regression of breakup day on early April ice thicknes")
slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(
    breakup.join(ice).dropna()['thick'], breakup.join(ice).dropna()['JulianDay'])
ax = plt.gca()
plt.text(.05, .95, f"slope={slope:.2f}±{std_err:.2f}, R^2 = {r_value**2:.3f}, p = {p_value**2:.6f}",
         ha='left', va='top', transform=ax.transAxes)


### All sites

In [ ]:
breakup = pd.read_csv(breakupdata, skiprows=3, index_col=0)
ice = pd.read_csv(icedata).set_index('year')

In [ ]:
results = []

for loc in breakup.siteID.unique():
    brk = breakup.copy()
    brk = brk[brk.siteID==loc]
    brk = brk[['year', 'JulianDay']].set_index('year')

    fig = plt.figure(figsize=(9, 6))
    sns.regplot(data=brk.join(ice).dropna(), x='thick', y='JulianDay')
    plt.title(f"{loc}: Regression of breakup day on early April Nenana ice thicknes")
    slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(
        brk.join(ice).dropna()['thick'], brk.join(ice).dropna()['JulianDay'])
    ax = plt.gca()
    plt.text(.05, .95, f"slope={slope:.2f}±{std_err:.2f}, R^2 = {r_value**2:.3f}, p = {p_value**2:.6f}",
         ha='left', va='top', transform=ax.transAxes)
    plt.show()

    results.append({
        "location": loc,
        "slope": slope,
        "R2":  r_value**2,
        "r_value":  r_value,
        "p_value": p_value,
        "std_err": std_err
    })

In [ ]:
pd.DataFrame.from_records(results)

In [ ]:
with open(outdir / "corr_IceThickNenana_breakupdate.csv", "w") as dst:
    dst.write(f"# Correlations from linear regression between last Nenana ice thickness and breakupdate \n")
    dst.write("# Data sent by Bob Busey, after Nenana Ice Classic site (cleaned up)\n")
    dst.write("# \n")
    pd.DataFrame.from_records(results).to_csv(dst)



### With detrending

In [ ]:
ice = pd.read_csv(icedata).set_index('year')
ice['thick'] = scipy.signal.detrend(ice.thick, type='linear')
ice

In [ ]:
results = []

for loc in breakup.siteID.unique():
    brk = breakup.copy()
    brk = brk[brk.siteID==loc]
    brk = brk[['year', 'JulianDay']].set_index('year')
    brk['JulianDay'] =  scipy.signal.detrend(brk['JulianDay'], type='linear')

    fig = plt.figure(figsize=(9, 6))
    sns.regplot(data=brk.join(ice).dropna(), x='thick', y='JulianDay')
    plt.title(f"{loc}: Regression of breakup day on early April Nenana ice thicknes")
    slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(
        brk.join(ice).dropna()['thick'], brk.join(ice).dropna()['JulianDay'])
    ax = plt.gca()
    plt.text(.05, .95, f"slope={slope:.2f}±{std_err:.2f}, R^2 = {r_value**2:.3f}, p = {p_value**2:.6f}",
         ha='left', va='top', transform=ax.transAxes)
    plt.show()

    results.append({
        "location": loc,
        "slope": slope,
        "R2":  r_value**2,
        "r_value":  r_value,
        "p_value": p_value,
        "std_err": std_err
    })

In [ ]:
pd.DataFrame.from_records(results)

In [ ]:
with open(outdir / "corr_IceThickNenana_breakupdate_detrended.csv", "w") as dst:
    dst.write(f"# Correlations from linear regression between last Nenana ice thickness and breakupdate, linear trends removed \n")
    dst.write("# Data sent by Bob Busey, after Nenana Ice Classic site (cleaned up)\n")
    dst.write("# \n")
    pd.DataFrame.from_records(results).to_csv(dst)
